- Thông tin thêm về data: https://huggingface.co/datasets/uitnlp/vietnamese_students_feedback
- Github data: https://github.com/buyamm/Sentiment-Analysis-VKU

In [1]:
import pandas as pd
import re
from nltk import ngrams

In [2]:
df = pd.read_csv("/kaggle/input/datasets/thngonquang/students-feedback/vietnamese_students_feedback_train.csv").dropna()
df_test = pd.read_csv("/kaggle/input/datasets/thngonquang/students-feedback/vietnamese_students_feedback_test.csv").dropna()
df_validation = pd.read_csv("/kaggle/input/datasets/thngonquang/students-feedback/vietnamese_students_feedback_validation.csv").dropna()

In [3]:
print(len(df))
print(len(df_test))
print(len(df_validation))

11426
3166
1583


In [4]:
print(df.head(5))
print(df_test.head(5))
print(df_validation.head(5))

                                            sentence  sentiment  topic
0                          slide giáo trình đầy đủ .          2      1
1     nhiệt tình giảng dạy , gần gũi với sinh viên .          2      0
2               đi học đầy đủ full điểm chuyên cần .          0      1
3  chưa áp dụng công nghệ thông tin và các thiết ...          0      0
4  thầy giảng bài hay , có nhiều bài tập ví dụ ng...          2      0
                                            sentence  sentiment  topic
0                           nói tiếng anh lưu loát .          2      0
1                           giáo viên rất vui tính .          2      0
2                                    cô max có tâm .          2      0
3                       giảng bài thu hút , dí dỏm .          2      0
4  giáo viên không giảng dạy kiến thức , hướng dẫ...          0      0
                                            sentence  sentiment  topic
0                           giáo trình chưa cụ thể .          0      1
1     

In [5]:
print("sentiment", df['sentiment'].unique())
print("topic: ", df['topic'].unique())

sentiment [2 0 1]
topic:  [1 0 3 2]


In [6]:
print("sentiment", df['sentiment'].value_counts())
print("topic: ", df['topic'].value_counts())

# sentiment: 0 (negative), 1 (neutral) and 2 (positive).
# topic: 0 (lecturer), 1 (training_program), 2 (facility) and 3 (others).

sentiment sentiment
2    5643
0    5325
1     458
Name: count, dtype: int64
topic:  topic
0    8166
1    2201
3     562
2     497
Name: count, dtype: int64


In [7]:
# Độ dài ký tự của mỗi dòng
df['length'] = df['sentence'].astype(str).apply(len)

# Min, Max
print("Min:", df['length'].min())
print("Max:", df['length'].max())

Min: 4
Max: 660


In [8]:
df['sentence'].str.len().describe()

count    11426.000000
mean        59.084894
std         43.085202
min          4.000000
25%         31.000000
50%         47.000000
75%         73.000000
max        660.000000
Name: sentence, dtype: float64

In [9]:
# df_sample.to_csv("viewdata.csv", sep='\t', encoding='utf-8')

# Làm sạch data

In [10]:
!pip install underthesea

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 88.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 56.2 MB/s eta 0:00:00


In [11]:
import unicodedata
import re
from underthesea import word_tokenize
stopwords = open('/kaggle/input/datasets/thngonquang/vietnamese-stopwords/vietnamese-stopwords-dash.txt', encoding='utf-8').read().splitlines()

In [12]:
# stopwords

In [13]:
def preprocess(text):
    text = unicodedata.normalize('NFC', text)
    text = text.lower()
    
    # slang normalize
    slang_dict = {
        "gv": "giảng viên",
        "sv": "sinh viên",
        "hk": "học kỳ"
    }
    for k, v in slang_dict.items():
        text = text.replace(k, v)
    
    # giữ emoji cơ bản
    text = re.sub(r'[^0-9a-zA-ZÀ-ỹ\s:()!?.]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    
    text = word_tokenize(text, format="text")
    
    return text

In [14]:
X_train = df['sentence']
X_test = df_test['sentence']
X_validation = df_validation['sentence']

y_train = df['sentiment']
y_test = df_test['sentiment']
y_validation = df_validation['sentiment']

In [15]:
X_train = X_train.apply(preprocess)
X_test = X_test.apply(preprocess)
X_validation = X_validation.apply(preprocess)

In [16]:
print(X_train.head(5))
print(X_test.head(5))
print(X_validation.head(5))

0                            slide giáo_trình đầy_đủ .
1         nhiệt_tình giảng_dạy gần_gũi với sinh_viên .
2                 đi học đầy_đủ full_điểm chuyên cần .
3    chưa áp_dụng công_nghệ_thông_tin và các thiết_...
4    thầy giảng bài hay có nhiều bài_tập ví_dụ ngay...
Name: sentence, dtype: object
0                             nói tiếng anh lưu_loát .
1                             giáo_viên rất vui_tính .
2                                      cô max có tâm .
3                           giảng bài thu_hút dí_dỏm .
4    giáo_viên không giảng_dạy kiến_thức hướng_dẫn ...
Name: sentence, dtype: object
0                             giáo_trình chưa cụ_thể .
1                                     giảng buồn_ngủ .
2                         giáo_viên vui_tính tận_tâm .
3    giảng_viên nên giao bài_tập nhiều hơn chia nhó...
4    giảng_viên cần giảng bài chi_tiết hơn đi_sâu h...
Name: sentence, dtype: object


# **TFIDF + SVM**

In [17]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix 

tfidf = TfidfVectorizer(ngram_range=(1, 2))
X_train_vectorizer_tfidf = tfidf.fit_transform(X_train).toarray()
X_test_vectorizer = tfidf.transform(X_test).toarray()

In [20]:
from sklearn.svm import LinearSVC
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

label_names = ['negative', 'neutral', 'positive']

# model
svm_model = LinearSVC()
svm_model.fit(X_train_vectorizer_tfidf, y_train)

# Predict
y_pred_svm = svm_model.predict(X_test_vectorizer)

svm_report = classification_report(y_test, y_pred_svm, output_dict=True)
print('=== TF-IDF + LinearSVC ===')
print(f"Accuracy: {accuracy_score(y_test, y_pred_svm):.4f}")
print(classification_report(y_test, y_pred_svm, target_names=label_names))


## **BoW + LogisticRegression**

In [21]:
from sklearn.feature_extraction.text import CountVectorizer

bow = CountVectorizer(ngram_range = (1,2))
X_train_vectorizer_bow = bow.fit_transform(X_train).toarray()
X_test_vectorizer = bow.transform(X_test).toarray()

In [23]:
from sklearn.linear_model import LogisticRegression

# model
lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train_vectorizer_bow, y_train)

# Predict
y_pred_lr = lr_model.predict(X_test_vectorizer)

lr_report = classification_report(y_test, y_pred_lr, output_dict=True)
print('=== BoW + Logistic Regression ===')
print(f"Accuracy: {accuracy_score(y_test, y_pred_lr):.4f}")
print(classification_report(y_test, y_pred_lr, target_names=label_names))


# ════════════════════════════════════════════════════════════════════════════
# SO SÁNH 2 MODEL
# ════════════════════════════════════════════════════════════════════════════

models_info = {
    'TF-IDF + LinearSVC':      (svm_report,  y_pred_svm),
    'BoW + LogisticRegression': (lr_report,   y_pred_lr),
}

# ── 1. Bảng tổng hợp metrics ─────────────────────────────────────────────
import pandas as pd
summary = pd.DataFrame([
    {
        'Model':         name,
        'Accuracy':      r['accuracy'],
        'Precision':     r['weighted avg']['precision'],
        'Recall':        r['weighted avg']['recall'],
        'F1 (weighted)': r['weighted avg']['f1-score'],
        'F1 (macro)':    r['macro avg']['f1-score'],
    }
    for name, (r, _) in models_info.items()
])
print('\n', summary.to_string(index=False))

# ── 2. Bar chart so sánh metrics ─────────────────────────────────────────
metrics_cols = ['Accuracy', 'Precision', 'Recall', 'F1 (weighted)', 'F1 (macro)']
x      = np.arange(len(metrics_cols))
width  = 0.32
colors = ['#3498db', '#e74c3c']

fig, ax = plt.subplots(figsize=(11, 5))
for i, (name, (r, _)) in enumerate(models_info.items()):
    scores = summary[summary['Model'] == name][metrics_cols].values[0]
    offset = (i - 0.5) * width
    bars = ax.bar(x + offset, scores, width, label=name, color=colors[i], alpha=0.85)
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
                f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels(metrics_cols, fontsize=10)
ax.set_ylim(0, 1.12)
ax.set_ylabel('Score', fontsize=11)
ax.set_title('So sánh TF-IDF + LinearSVC vs BoW + Logistic Regression', fontsize=12)
ax.legend(fontsize=10)
ax.yaxis.grid(True, linestyle='--', alpha=0.6)
ax.set_axisbelow(True)
plt.tight_layout()
plt.show()

# ── 3. Per-class F1 heatmap ───────────────────────────────────────────────
per_class = pd.DataFrame(
    {name: [r[str(i)]['f1-score'] for i in range(3)] for name, (r, _) in models_info.items()},
    index=label_names
)
fig, ax = plt.subplots(figsize=(6, 3))
sns.heatmap(per_class, annot=True, fmt='.3f', cmap='YlGnBu',
            vmin=0, vmax=1, linewidths=0.5, ax=ax)
ax.set_title('Per-class F1-score theo từng model', fontsize=11)
ax.set_xlabel('Model')
ax.set_ylabel('Class')
plt.tight_layout()
plt.show()

# ── 4. Confusion matrix song song ────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#f8f9fa')

for ax, (name, (r, y_pred)) in zip(axes, models_info.items()):
    cm = confusion_matrix(y_test, y_pred)
    cm_norm = cm.astype('float') / cm.sum(axis=1, keepdims=True)

    # Custom annotation: "count\n(pct%)"
    raw_cm = confusion_matrix(y_test, y_pred)
    annots = np.empty_like(raw_cm, dtype=object)
    for i in range(raw_cm.shape[0]):
        for j in range(raw_cm.shape[1]):
            annots[i, j] = f"{raw_cm[i,j]}\n({cm_norm[i,j]:.0%})"

    ax.set_facecolor('#f8f9fa')
    sns.heatmap(
        cm_norm, ax=ax,
        annot=annots, fmt='', cmap='Blues',
        xticklabels=label_names, yticklabels=label_names,
        vmin=0, vmax=1,
        linewidths=1.5, linecolor='white',
        cbar_kws={'shrink': 0.8, 'label': 'Tỉ lệ'},
        annot_kws={'size': 12, 'weight': 'bold'},
    )

    # Highlight diagonal (correct predictions)
    for i in range(len(label_names)):
        ax.add_patch(plt.Rectangle((i, i), 1, 1, fill=False,
                                   edgecolor='#2ecc71', lw=2.5))

    ax.set_title(f'{name}', fontsize=12, fontweight='bold', pad=12)
    ax.set_ylabel('Thực tế (Actual)', fontsize=10, labelpad=8)
    ax.set_xlabel('Dự đoán (Predicted)', fontsize=10, labelpad=8)
    ax.tick_params(axis='both', labelsize=10)
    ax.set_yticklabels(ax.get_yticklabels(), rotation=0)

fig.suptitle('Confusion Matrix – So sánh 2 Model', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()
